# prep

In [ ]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [ ]:
# Data prep
query = f"""

SELECT security_isin
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state f
WHERE security_type IN ('GOVS', 'FIDE')
AND assttp_scty_issr_sector_riad = 'S1311'
AND business_date >= '2023-01-01' 
AND gnlcoll = 'SPEC'
GROUP BY security_isin
HAVING COUNT(DISTINCT security_type) = 2

"""

govs = pd.read_sql_query(query, cnxn)

In [ ]:
unique_govs = tuple(govs['security_isin'])

In [ ]:
ust = pd.read_csv('Data\\TreasuryCusip.csv')

In [ ]:
treasuries = tuple(ust['ISIN'].unique())

In [ ]:
# Data prep
query = f"""

SELECT
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END AS a,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END AS b
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
WHERE intragroup = 1
GROUP BY
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END;


"""

df_edges = pd.read_sql_query(query, cnxn)

In [ ]:
import pandas as pd

# edges: columns ['a','b']
edges = df_edges.values.tolist()

parent = {}
rank = {}

def find(x):
    if parent.setdefault(x, x) != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return
    rxr, ryr = rank.setdefault(rx,0), rank.setdefault(ry,0)
    if rxr < ryr:
        parent[rx] = ry
    elif rxr > ryr:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] = rxr + 1

for a,b in edges:
    union(a,b)

# canonical id = min-LEI per component (stable label)
from collections import defaultdict
comps = defaultdict(list)
for x in parent.keys():
    comps[find(x)].append(x)

canon = {rep: min(members) for rep, members in comps.items()}
mapping = []
for x in parent.keys():
    rep = find(x)
    mapping.append((x, canon[rep]))

df_mapping_cc = pd.DataFrame(mapping, columns=["lei","group_id_synth"]).sort_values(["group_id_synth","lei"])

In [ ]:
foreign_bg = ('0W1U67PTV5WY3WYWKD79',
'2138008P9NOMBRMROI73',
'213800A9GT65GAES2V60',
'213800G8QEXN34A2YG53',
'213800GVD8L87R18CT98',
'213800IBT39XQ9C4CP71',
'213800NW35DTWHTMX505',
'213800RZ3GCUXMBGYN59',
'2IGI19DL77OX0HC3ZE78',
'4PQUHN3JPFGFNF3BB653',
'4ZHCHI4KYZG2WVRT8631',
'5299007QVIQ7IO64NX37',
'5493000IQQ05Y25L0V92',
'5493000YPN33HF74SN02',
'54930010P7BUGOECPI58',
'5493002XYZZ0CGQ6CB58',
'54930040QPHHWGT1J432',
'54930050SE0SM7CM2G07',
'5493006PWI2H6PX25403',
'5493007VSMFZCPV1NB83',
'5493008GNQHVI377MY19',
'5493009NLZXZGJDOPC94',
'549300BKWHXYEXPV0328',
'549300FH0WJAPEHTIQ77',
'549300HU9EWFS3CNX640',
'549300KP56LL8NKKFL47',
'549300RSY622D5TQWS42',
'549300SXSTGQY3EA1B18',
'549300WDT1HWUMTUW770',
'571474TGEMMWANRLN572',
'724500AAT1DK36059L16',
'9676007O0UF5YB3QPR03',
'BWS7DNS2Z4NPKPNYKL75',
'HV5W8PGLJ127N2SFSM23'
)

In [ ]:
euro_area_bg = ('0IKLU6X1B10WK7X42C15',
'1VUV7VQFKUOQSJ21A208',
'2138001YBO7CJNQOEE37',
'21380027LW8AF6I1WA03',
'2138003Z5ZVN16GFYV70',
'2138004VZX8CSGPTDX68',
'21380073P7J4PAD91E29',
'213800AGKVL18YKQSV51',
'213800BWHAS44J2C1B28',
'213800D4LHBCXXEEE235',
'213800DBQIB6VBNU5C64',
'213800FKGFR7Q2ACLS83',
'213800G63T4ER4MSVR22',
'213800HV6TP2I5A6MW58',
'213800I92TAU7I3FP232',
'213800KGF4EFNUQKAT69',
'213800OOQOSULB37T658',
'213800ZIGVOZ992FNQ85',
'222100D7H9VRJEH7DU25',
'222100M2PU043YB7YQ06',
'23B6332KMR0JIZLJG565',
'2534006G7F7F1TFC9T77',
'2534006HF1L4YF10UD91',
'253400N2R0RF14JQ0060',
'2549002MVYWWDX54IB83',
'259400QHDOZWMJ103294',
'259400YLRTOBISHBVX41',
'2W8N8UU78PMDQKZENC08',
'31570010000000036567',
'315700GBLUBZ50S45F53',
'315700GXRKZ452JF2U13',
'3M5E1GQGKL17HI6CPN30',
'3U8WV1YX2VMUHH7Z1Q21',
'48510000156ESYOBV122',
'52965FONQ5NZKP0WZL45',
'52990002O5KK6XOGJ020',
'52990010C4NK412ZL440',
'5299004TE2DYMKEAM814',
'5299005UJX6K7BQKV086',
'52990080NNXXLC14OC65',
'529900AQBND3S6YJLY83',
'529900C214QOT3ZYD838',
'529900C4RSSBWXBSY931',
'529900E1WHT64CB20277',
'529900FWTU88V844B672',
'529900GGYMNGRQTDOO93',
'529900K16YGKC8BES892',
'529900T32UL0CP1FZA06',
'529900UKZBMDBDZIXD62',
'529900VA5CNBWXAONR25',
'549300298FD7AS4PPU70',
'54930056IRBXK0Q1FP96',
'549300685QG7DJS55M76',
'5493007RT80TMKY7TO30',
'549300ABE4K96QOCEH37',
'549300CQ9NLEHMRCU505',
'549300DV870NBWY5W279',
'549300DY78U4CMKNHE48',
'549300DYPOFMXOR7XM56',
'549300FOF121DSRG5867',
'549300FR956J8UJDWQ78',
'549300GOF5A5DHWJLV03',
'549300GRXFI7D6PNEA68',
'549300L7YCATGO57ZE10',
'549300LYFYVPUCG6SY25',
'549300NC3SZTETC10349',
'549300NEBDPH0ZXIF850',
'549300Z6OB1D4ZUBD145',
'63540061DPCBNMCGRY22',
'635400CE9HHFB55PEY43',
'635400GQWXFJDCQXW612',
'635400LNHEPZBRNB5D58',
'635400LRAHYBRUZCIH13',
'724500A11NIP5HCVF984',
'815600154F8F91CF6B05',
'8156002070DA4DCBFF31',
'81560027D07F9BDB8436',
'81560038903FF9FA8F80',
'815600522538355AE429',
'8156007395B20763EB44',
'8156009F40F6523F7022',
'815600A32DA05F693F24',
'8EFE15WY4PBBKG6GZI21',
'95980020140005184148',
'9598009X93GQBNHF1L85',
'959800T0J9ADL4GYSS41',
'9695000O0HNLFNR0FB30',
'969500DCEVPV6UIYK220',
'969500QLO3GN6GUB4P61',
'FI6C7E5PBUB3F9K43B44',
'GP5DT10VX1QRQUKVBK64',
'J4CP7MHCXR8DAQMKIL78',
'NNVPP80YIZGEY2314M97',
'RRAN7P32P0W0YY4XQW79',
'SI5RG2M0WQQLZCXKRM20'
)

In [ ]:
bgroups = tuple(set(foreign_bg + euro_area_bg))

In [ ]:
foreign_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(foreign_bg), 'lei'].unique())
euro_area_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(euro_area_bg), 'lei'].unique())

# confidentiality rule

In [ ]:
# At least three entities (distinct LEIs) behind every displayed value, and no single entity above 85 percent of it.
# Contributions are absolute volumes, so net positions are checked on gross volumes.
def check(df, by, entity='entity_id', value='value'):
    d = df.copy()
    d[value] = d[value].abs()
    c = d.groupby(by + [entity], as_index=False)[value].sum()
    c = c[c[value] > 0]
    c['share'] = c[value] / c.groupby(by)[value].transform('sum')
    res = c.groupby(by).agg(n_entity=(entity, 'nunique'), max_entity=('share', 'max')).reset_index()
    res['ok'] = (res['n_entity'] >= 3) & (res['max_entity'] <= 0.85)
    return res

In [ ]:
# collateral filters, identical to the ones in the other notebooks
coll_eur = f"""
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
"""
coll_usd = f"""
AND s.security_isin IN {treasuries}
AND s.gnlcoll = 'SPEC'
"""
ea = "('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')"

# Table 1: Branch-Subsidiary

## EUR

In [ ]:
query = f"""

SELECT lender_id AS entity_id,
  CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
  CASE
    WHEN s.lender_country IN ('US') THEN 'US'
    WHEN s.lender_country IN ('GB') THEN 'UK'
    WHEN s.lender_country IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS country_group,
  SUM(nominal_value) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
  AND nominal_ccy = 'EUR'
  AND intragroup = 1
  AND central_clearing = 'non-cleared'
  {coll_eur}
GROUP BY lender_id, residence_group, country_group

"""

df_lend = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT borrower_id AS entity_id,
  CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
  CASE
    WHEN s.borrower_country IN ('US') THEN 'US'
    WHEN s.borrower_country IN ('GB') THEN 'UK'
    WHEN s.borrower_country IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS country_group,
  SUM(nominal_value) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
  AND nominal_ccy = 'EUR'
  AND intragroup = 1
  AND central_clearing = 'non-cleared'
  {coll_eur}
GROUP BY borrower_id, residence_group, country_group

"""

df_borr = pd.read_sql_query(query, cnxn)

In [ ]:
df = pd.concat([df_lend, df_borr])
df['entity_type'] = np.where(df['residence_group'] == df['country_group'], 'Subsidiary', 'Branch')
check(df, ['residence_group', 'entity_type'])

## USD

In [ ]:
query = f"""

SELECT lender_id AS entity_id,
  CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
  CASE
    WHEN s.lender_country IN ('US') THEN 'US'
    WHEN s.lender_country IN ('GB') THEN 'UK'
    WHEN s.lender_country IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS country_group,
  SUM(nominal_value) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
  AND nominal_ccy = 'USD'
  AND intragroup = 1
  AND central_clearing = 'non-cleared'
  {coll_usd}
GROUP BY lender_id, residence_group, country_group

"""

df_lend = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT borrower_id AS entity_id,
  CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
  CASE
    WHEN s.borrower_country IN ('US') THEN 'US'
    WHEN s.borrower_country IN ('GB') THEN 'UK'
    WHEN s.borrower_country IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS country_group,
  SUM(nominal_value) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
  AND nominal_ccy = 'USD'
  AND intragroup = 1
  AND central_clearing = 'non-cleared'
  {coll_usd}
GROUP BY borrower_id, residence_group, country_group

"""

df_borr = pd.read_sql_query(query, cnxn)

In [ ]:
df = pd.concat([df_lend, df_borr])
df['entity_type'] = np.where(df['residence_group'] == df['country_group'], 'Subsidiary', 'Branch')
check(df, ['residence_group', 'entity_type'])

# Table 2: Market clearing structure

## EUR

In [ ]:
query = f"""

SELECT central_clearing, intragroup, lender_id, borrower_id,
  s_lender.sector AS lender_sector, s_borrower.sector AS borrower_sector,
  SUM(nominal_euro) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date > '2021-01-04' AND business_date <= '2025-07-01'
{coll_eur}
GROUP BY central_clearing, intragroup, lender_id, borrower_id, s_lender.sector, s_borrower.sector

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
# three rows of the table: total, intragroup, interbank. Lender side and borrower side checked separately.
# In cleared trades the CCP is on one side, which shows up as a large single share on that side.
df['row'] = 'Total volume'
intra = df[df['intragroup'] == 1].assign(row='Intragroup')
inter = df[(df['intragroup'] == 0) & (df['lender_sector'] == 'BANK') & (df['borrower_sector'] == 'BANK')].assign(row='Interbank')
rows = pd.concat([df, intra, inter])

In [ ]:
check(rows, ['central_clearing', 'row'], entity='lender_id')

In [ ]:
check(rows, ['central_clearing', 'row'], entity='borrower_id')

## USD

In [ ]:
query = f"""

SELECT central_clearing, intragroup, lender_id, borrower_id,
  s_lender.sector AS lender_sector, s_borrower.sector AS borrower_sector,
  SUM(nominal_euro) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date > '2021-01-04' AND business_date <= '2025-07-01'
{coll_usd}
GROUP BY central_clearing, intragroup, lender_id, borrower_id, s_lender.sector, s_borrower.sector

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
# three rows of the table: total, intragroup, interbank. Lender side and borrower side checked separately.
# In cleared trades the CCP is on one side, which shows up as a large single share on that side.
df['row'] = 'Total volume'
intra = df[df['intragroup'] == 1].assign(row='Intragroup')
inter = df[(df['intragroup'] == 0) & (df['lender_sector'] == 'BANK') & (df['borrower_sector'] == 'BANK')].assign(row='Interbank')
rows = pd.concat([df, intra, inter])

In [ ]:
check(rows, ['central_clearing', 'row'], entity='lender_id')

In [ ]:
check(rows, ['central_clearing', 'row'], entity='borrower_id')

# Table 3: Jurisdictions (nationality)

## EUR

In [ ]:
# same CASE logic as the original query, each trade goes to the first matching bucket
query = f"""

SELECT lender_id, borrower_id,
CASE 
  WHEN (lender_country_nationality IN {ea} AND borrower_country_nationality IN {ea})
    OR lender_name IN ('BNP Paribas Fortis', 'BNP PARIBAS', 'SOCIETE GENERALE INTERNATIONAL LIMITED') THEN 'Euro area'
  WHEN ((lender_country_nationality IN {ea} AND borrower_country_nationality IN ('US'))
    OR (lender_country_nationality IN ('US') AND borrower_country_nationality IN {ea}))
    AND lender_name NOT IN ('BNP Paribas Fortis', 'BNP PARIBAS') THEN 'US'
  WHEN ((lender_country_nationality IN {ea} AND borrower_country_nationality IN ('GB'))
    OR (lender_country_nationality IN ('GB') AND borrower_country_nationality IN {ea}))
    AND lender_name NOT IN ('SOCIETE GENERALE INTERNATIONAL LIMITED') THEN 'UK'
  ELSE 'Other'
END AS nationality,
SUM(nominal_euro) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s
WHERE central_clearing = 'non-cleared'
AND intragroup = 1
AND nominal_ccy = 'EUR'
AND valdt > '2021-01-04' AND valdt <= '2025-07-01'
{coll_eur}
GROUP BY lender_id, borrower_id, nationality

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
check(df, ['nationality'], entity='lender_id')

In [ ]:
check(df, ['nationality'], entity='borrower_id')

## USD

In [ ]:
query = f"""

SELECT lender_id, borrower_id,
CASE 
  WHEN (lender_country_nationality IN {ea} AND borrower_country_nationality IN {ea})
    OR lender_name IN ('CREDIT AGRICOLE CORPORATE AND INVESTMENT BANK', 'BNP PARIBAS', 'BANCO BILBAO VIZCAYA ARGENTARIA SOCIEDAD ANONIMA', 'BBVA SECURITIES INC.', 'CREDIT AGRICOLE SECURITIES (USA) INC.')
    OR lender_id IN ('R0MUWSFPU8MPRO8K5P83', 'RCNB6OTYUAMMP879YW96') THEN 'Euro area'
  WHEN ((lender_country_nationality IN {ea} AND borrower_country_nationality IN ('US'))
    OR (lender_country_nationality IN ('US') AND borrower_country_nationality IN {ea}))
    AND lender_name NOT IN ('CREDIT AGRICOLE CORPORATE AND INVESTMENT BANK', 'BNP PARIBAS', 'BANCO BILBAO VIZCAYA ARGENTARIA SOCIEDAD ANONIMA', 'BBVA SECURITIES INC.', 'CREDIT AGRICOLE SECURITIES (USA) INC.')
    AND lender_id NOT IN ('R0MUWSFPU8MPRO8K5P83', 'RCNB6OTYUAMMP879YW96') THEN 'US'
  WHEN (lender_country_nationality IN {ea} AND borrower_country_nationality IN ('GB'))
    OR (lender_country_nationality IN ('GB') AND borrower_country_nationality IN {ea}) THEN 'UK'
  ELSE 'Other'
END AS nationality,
SUM(nominal_euro) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s
WHERE central_clearing = 'non-cleared'
AND intragroup = 1
AND nominal_ccy = 'USD'
AND valdt > '2021-01-04' AND valdt <= '2025-07-01'
{coll_usd}
GROUP BY lender_id, borrower_id, nationality

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
check(df, ['nationality'], entity='lender_id')

In [ ]:
check(df, ['nationality'], entity='borrower_id')

# Figure 1: Net positions by residence

## EUR

In [ ]:
query = f"""

SELECT s.business_date, lender_id AS entity_id,
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group, 
  sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND intragroup = 1
{coll_eur}
GROUP BY s.business_date, lender_id, residence_group

"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT s.business_date, borrower_id AS entity_id,
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group, 
  sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND intragroup = 1
{coll_eur}
GROUP BY s.business_date, borrower_id, residence_group

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# every date is a displayed value, so the check is per date and residence group
res = check(pd.concat([df_lending, df_borrowing]), ['business_date', 'residence_group'])
res.groupby('residence_group').agg(dates=('ok', 'size'), violations=('ok', lambda x: (~x).sum()), min_n_entity=('n_entity', 'min'), max_entity=('max_entity', 'max'))

In [ ]:
res[~res['ok']]

## USD

In [ ]:
query = f"""

SELECT s.business_date, lender_id AS entity_id,
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group, 
  sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01'
AND nominal_ccy IN ('USD')
AND central_clearing = 'non-cleared'
AND intragroup = 1
{coll_usd}
GROUP BY s.business_date, lender_id, residence_group

"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT s.business_date, borrower_id AS entity_id,
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN {ea} THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group, 
  sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01'
AND nominal_ccy IN ('USD')
AND central_clearing = 'non-cleared'
AND intragroup = 1
{coll_usd}
GROUP BY s.business_date, borrower_id, residence_group

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# every date is a displayed value, so the check is per date and residence group
res = check(pd.concat([df_lending, df_borrowing]), ['business_date', 'residence_group'])
res.groupby('residence_group').agg(dates=('ok', 'size'), violations=('ok', lambda x: (~x).sum()), min_n_entity=('n_entity', 'min'), max_entity=('max_entity', 'max'))

In [ ]:
res[~res['ok']]

# Chains EUR: euro area subsidiaries of foreign groups (Figure 2a, Tables 5 to 8)

Same legs as chains.ipynb cells 72 to 86 and 106, with the residence restriction in SQL. One row per date, entity and security.

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as cleared_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 0
AND central_clearing = 'cleared'
AND lender_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, lender_id, security_isin
  
"""

df_cleared_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
sum(nominal_value) as cleared_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 0
AND central_clearing = 'cleared'
AND borrower_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, borrower_id, security_isin
  
"""

df_cleared_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND lender_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, lender_id, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND borrower_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, borrower_id, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
keys = ['business_date', 'entity_id', 'security_isin']
df = (df_cleared_lending.merge(df_cleared_borrowing, on=keys, how='outer')
                        .merge(df_intra_lending, on=keys, how='outer')
                        .merge(df_intra_borrowing, on=keys, how='outer')).fillna(0)
df['cleared_net'] = df['cleared_lending'] - df['cleared_borrowing']
df['intra_net'] = df['intra_lending'] - df['intra_borrowing']
df['cleared_gross'] = df['cleared_lending'] + df['cleared_borrowing']
df['intra_gross'] = df['intra_lending'] + df['intra_borrowing']
df['matched'] = ((df['cleared_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['cleared_lending'] > 0) & (df['intra_borrowing'] > 0))
df['chain_net'] = np.minimum(df['intra_borrowing'], df['cleared_lending']) - np.minimum(df['cleared_borrowing'], df['intra_lending'])

## Figure 2a: net cleared and net intragroup per date

In [ ]:
res = check(df, ['business_date'], value='cleared_gross')
print(f"net cleared: {len(res)} dates, {(~res['ok']).sum()} violations")
res = check(df, ['business_date'], value='intra_gross')
print(f"net intragroup: {len(res)} dates, {(~res['ok']).sum()} violations")

## Tables 5 and 6: numerators and denominators of the bounds

In [ ]:
# every bound is a ratio of sums over the panel, each sum is checked on its own
sums = pd.concat([
    df.assign(stat='lower numerator |chain_net|', value=df['chain_net']),
    df.assign(stat='intra denominator |intra_net|', value=df['intra_net']),
    df[df['matched']].assign(stat='upper numerator intra, matched |intra_net|', value=df['intra_net']),
    df.assign(stat='cleared denominator |cleared_net|', value=df['cleared_net']),
    df[df['matched']].assign(stat='upper numerator cleared, matched |cleared_net|', value=df['cleared_net']),
])
check(sums, ['stat'])

## Table 8: pass-through sample

In [ ]:
check(pd.concat([df.assign(stat='gross intragroup', value=df['intra_gross']),
                 df.assign(stat='gross cleared', value=df['cleared_gross'])]), ['stat'])

## Table 7: unconditional denominator, all euro area entities in the cleared segment

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as cleared_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND central_clearing = 'cleared'
AND lender_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, lender_id, security_isin
  
"""

df_all_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
sum(nominal_value) as cleared_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND central_clearing = 'cleared'
AND borrower_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, borrower_id, security_isin
  
"""

df_all_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
df_all = df_all_lending.merge(df_all_borrowing, on=keys, how='outer').fillna(0)
df_all['value'] = df_all['cleared_lending'] - df_all['cleared_borrowing']
check(df_all.assign(stat='unconditional cleared denominator |cleared_net|'), ['stat'])

# Chains USD: UK branches of euro area groups (Figure 2b, Tables 5 to 8)

Same legs as chains.ipynb cells 39 to 53 and 105. The fund's residence is kept because Tables 5 to 7 use all investment funds while Table 8 and Figure 2 use Cayman funds only. The hedge fund legs keep the fund id to check the fund side as well.

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, borrower_id as fund_id, borrower_country_residence as fund_residence, security_isin, 
sum(nominal_value) as if_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND lender_id in {euro_area_entity}
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_borrower.sector = 'IF'
AND lender_country_residence = 'GB'
{coll_usd}
GROUP BY business_date, lender_id, borrower_id, borrower_country_residence, security_isin
  
"""

df_if_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, lender_id as fund_id, lender_country_residence as fund_residence, security_isin, 
sum(nominal_value) as if_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND borrower_id in {euro_area_entity}
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_lender.sector = 'IF'
AND borrower_country_residence = 'GB'
{coll_usd}
GROUP BY business_date, borrower_id, lender_id, lender_country_residence, security_isin
  
"""

df_if_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND lender_id in {euro_area_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND lender_country_residence = 'GB'
{coll_usd}
GROUP BY business_date, lender_id, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND borrower_id in {euro_area_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND borrower_country_residence = 'GB'
{coll_usd}
GROUP BY business_date, borrower_id, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
keys = ['business_date', 'entity_id', 'security_isin']

def usd_panel(if_l, if_b):
    d = (if_l.groupby(keys, as_index=False)['if_lending'].sum()
             .merge(if_b.groupby(keys, as_index=False)['if_borrowing'].sum(), on=keys, how='outer')
             .merge(df_intra_lending, on=keys, how='outer')
             .merge(df_intra_borrowing, on=keys, how='outer')).fillna(0)
    d['if_net'] = d['if_lending'] - d['if_borrowing']
    d['intra_net'] = d['intra_lending'] - d['intra_borrowing']
    d['if_gross'] = d['if_lending'] + d['if_borrowing']
    d['intra_gross'] = d['intra_lending'] + d['intra_borrowing']
    d['matched'] = ((d['if_borrowing'] > 0) & (d['intra_lending'] > 0)) | ((d['if_lending'] > 0) & (d['intra_borrowing'] > 0))
    d['chain_net'] = np.minimum(d['intra_borrowing'], d['if_lending']) - np.minimum(d['if_borrowing'], d['intra_lending'])
    return d

df = usd_panel(df_if_lending, df_if_borrowing)                                                          # all investment funds, Tables 5 to 7
df_ky = usd_panel(df_if_lending[df_if_lending['fund_residence'] == 'KY'], df_if_borrowing[df_if_borrowing['fund_residence'] == 'KY'])   # Cayman only, Table 8 and Figure 2b

## Figure 2b: net vs hedge funds and net intragroup per date

In [ ]:
res = check(df_ky, ['business_date'], value='if_gross')
print(f"net vs hedge funds: {len(res)} dates, {(~res['ok']).sum()} violations")
res = check(df_ky, ['business_date'], value='intra_gross')
print(f"net intragroup: {len(res)} dates, {(~res['ok']).sum()} violations")

## Tables 5 and 6

In [ ]:
sums = pd.concat([
    df.assign(stat='lower numerator |chain_net|', value=df['chain_net']),
    df.assign(stat='intra denominator |intra_net|', value=df['intra_net']),
    df[df['matched']].assign(stat='upper numerator intra, matched |intra_net|', value=df['intra_net']),
    df.assign(stat='hf denominator |if_net|', value=df['if_net']),
    df[df['matched']].assign(stat='upper numerator hf, matched |if_net|', value=df['if_net']),
])
check(sums, ['stat'])

In [ ]:
# fund side of the hedge fund leg
funds = pd.concat([df_if_lending.rename(columns={'if_lending': 'value'}), df_if_borrowing.rename(columns={'if_borrowing': 'value'})])
check(funds.assign(stat='all investment funds'), ['stat'], entity='fund_id')

## Table 8: pass-through sample (Cayman funds)

In [ ]:
check(pd.concat([df_ky.assign(stat='gross intragroup', value=df_ky['intra_gross']),
                 df_ky.assign(stat='gross hedge fund', value=df_ky['if_gross'])]), ['stat'])

In [ ]:
check(funds[funds['fund_residence'] == 'KY'].assign(stat='Cayman funds'), ['stat'], entity='fund_id')

## Table 7: unconditional denominator, all UK entities against investment funds

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as if_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_borrower.sector = 'IF'
AND lender_country_residence = 'GB'
{coll_usd}
GROUP BY business_date, lender_id, security_isin
  
"""

df_all_lending = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
sum(nominal_value) as if_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_lender.sector = 'IF'
AND borrower_country_residence = 'GB'
{coll_usd}
GROUP BY business_date, borrower_id, security_isin
  
"""

df_all_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
df_all = df_all_lending.merge(df_all_borrowing, on=keys, how='outer').fillna(0)
df_all['value'] = df_all['if_lending'] - df_all['if_borrowing']
check(df_all.assign(stat='unconditional hf denominator |if_net|'), ['stat'])

# Table 4: Probability of hedge fund activity

Three sets of entities enter: the intragroup flag (euro area subsidiaries of foreign groups borrowing intragroup), the hedge fund leg (euro area entities lending to Cayman funds), and the cleared universe. Sample from 2021-01-04 as in chains.ipynb cells 13 to 37.

In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, lender_id as cpty_id, security_isin, 
sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND borrower_country_residence IN {ea}
{coll_eur}
GROUP BY business_date, borrower_id, lender_id, security_isin
  
"""

df_flag = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, borrower_id as fund_id, security_isin, 
sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {euro_area_entity}
AND intragroup = 0
AND central_clearing = 'non-cleared'
AND s_borrower.sector = 'IF' AND borrower_country_residence = 'KY'
{coll_eur}
GROUP BY business_date, lender_id, borrower_id, security_isin
  
"""

df_hf = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT lender_id as entity_id, borrower_id as cpty_id,
sum(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND central_clearing = 'cleared'
{coll_eur}
GROUP BY lender_id, borrower_id
  
"""

df_universe = pd.read_sql_query(query, cnxn)

In [ ]:
chain = df_flag[['business_date', 'security_isin']].drop_duplicates().assign(column='Intragroup securities')
df_hf = df_hf.merge(chain, on=['business_date', 'security_isin'], how='left')
df_hf['column'] = df_hf['column'].fillna('Non intragroup securities')

In [ ]:
check(df_flag.assign(stat='intragroup flag, subsidiaries'), ['stat'])

In [ ]:
check(df_hf, ['column'])

In [ ]:
check(df_hf, ['column'], entity='fund_id')

In [ ]:
# cleared universe, the CCP is on one side of every trade
check(df_universe.assign(stat='cleared universe, lenders'), ['stat'])

# Table 9 and appendix: Specialness regressions

Same legs as market_impact.ipynb cells 14 to 33. The regression is bond by date, the displayed values are coefficients. The check reports the concentration of each regressor and the dependent variable over the estimation sample. Cleared matched and gross versions use the same contributors.

In [ ]:
query = f"""

SELECT business_date, security_isin, lender_id as bank_id,
sum(nominal_value)/1e9 as cleared_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0 
AND central_clearing = 'cleared'
{coll_eur}
AND repo_rate IS NOT NULL
GROUP BY business_date, security_isin, bank_id
  
"""

df_cleared = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, security_isin, borrower_id as bank_id,
sum(nominal_value)/1e9 as intra_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 1 
AND borrower_country_residence IN {ea}
AND central_clearing = 'non-cleared'
{coll_eur}
GROUP BY business_date, security_isin, bank_id
  
"""

df_intra = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT business_date, security_isin, borrower_id as bank_id, lender_id as fund_id,
sum(nominal_value)/1e9 as hf_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND borrower_country_residence IN {ea}
AND lender_country_residence = 'KY'
AND central_clearing = 'non-cleared'
{coll_eur}
GROUP BY business_date, security_isin, bank_id, lender_id
  
"""

df_hf = pd.read_sql_query(query, cnxn)

In [ ]:
# linked = intragroup borrowing by banks that also lend in cleared that bond-day (the left merge in market_impact.ipynb)
k = ['business_date', 'security_isin', 'bank_id']
in_cleared = df_cleared[k].assign(in_cleared=True)
df_intra = df_intra.merge(in_cleared, on=k, how='left')
df_intra['in_cleared'] = df_intra['in_cleared'].fillna(False).astype(bool)
df_hf = df_hf.merge(in_cleared, on=k, how='inner')

# bank benchmark = cleared lending by banks with no intragroup and no hedge fund activity that bond-day
active = pd.concat([df_intra[k], df_hf[k]]).drop_duplicates().assign(active=True)
bench = df_cleared.merge(active, on=k, how='left')
bench = bench[bench['active'].isna()]

sums = pd.concat([
    df_cleared.assign(stat='dependent, cleared lenders', value=df_cleared['cleared_vol']),
    df_intra[df_intra['in_cleared']].assign(stat='intragroup linked', value=df_intra['intra_vol']),
    df_intra[~df_intra['in_cleared']].assign(stat='intragroup unlinked', value=df_intra['intra_vol']),
    df_hf.assign(stat='hf volume', value=df_hf['hf_vol']),
    bench.assign(stat='bank benchmark', value=bench['cleared_vol']),
])
check(sums, ['stat'], entity='bank_id')

In [ ]:
check(df_hf.assign(stat='hf volume, funds', value=df_hf['hf_vol']), ['stat'], entity='fund_id')

# Appendix table: bilateral flow direction

## EUR

In [ ]:
query = f"""

SELECT lender_id, borrower_id,
  CASE WHEN s.lender_country_residence IN {ea} THEN 'EA' ELSE s.lender_country_residence END AS lender_jur,
  CASE WHEN s.borrower_country_residence IN {ea} THEN 'EA' ELSE s.borrower_country_residence END AS borrower_jur,
  SUM(nominal_euro) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s 
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01' 
  AND s.nominal_ccy IN ('EUR') 
  AND s.central_clearing = 'non-cleared' 
  AND s.intragroup = 1 
  {coll_eur}
GROUP BY lender_id, borrower_id, lender_jur, borrower_jur

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
core = df['lender_jur'].isin(['EA', 'US', 'GB']) & df['borrower_jur'].isin(['EA', 'US', 'GB'])
df['cell'] = np.where(core, df['lender_jur'] + ' -> ' + df['borrower_jur'], 'Other')

In [ ]:
check(df, ['cell'], entity='lender_id')

In [ ]:
check(df, ['cell'], entity='borrower_id')

In [ ]:
# cross border share quoted in the text of Section 3
df['cell'] = np.where(df['lender_jur'] == df['borrower_jur'], 'Domestic', 'Cross border')
check(df, ['cell'], entity='lender_id')

## USD

In [ ]:
query = f"""

SELECT lender_id, borrower_id,
  CASE WHEN s.lender_country_residence IN {ea} THEN 'EA' ELSE s.lender_country_residence END AS lender_jur,
  CASE WHEN s.borrower_country_residence IN {ea} THEN 'EA' ELSE s.borrower_country_residence END AS borrower_jur,
  SUM(nominal_euro) AS value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s 
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01' 
  AND s.nominal_ccy IN ('USD') 
  AND s.central_clearing = 'non-cleared' 
  AND s.intragroup = 1 
  {coll_usd}
GROUP BY lender_id, borrower_id, lender_jur, borrower_jur

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
core = df['lender_jur'].isin(['EA', 'US', 'GB']) & df['borrower_jur'].isin(['EA', 'US', 'GB'])
df['cell'] = np.where(core, df['lender_jur'] + ' -> ' + df['borrower_jur'], 'Other')

In [ ]:
check(df, ['cell'], entity='lender_id')

In [ ]:
check(df, ['cell'], entity='borrower_id')

In [ ]:
# cross border share quoted in the text of Section 3
df['cell'] = np.where(df['lender_jur'] == df['borrower_jur'], 'Domestic', 'Cross border')
check(df, ['cell'], entity='lender_id')

# Appendix figure: Outstanding time series nationality

## EUR

In [ ]:
query = f"""

SELECT s.business_date, lender_id, borrower_id,
CASE 
    WHEN borrower_country_nationality IN ('US') OR lender_country_nationality IN ('US') THEN 'US'  
    WHEN borrower_country_nationality IN ('GB') OR lender_country_nationality IN ('GB') THEN 'GB'  
    WHEN borrower_country_nationality IN {ea} AND lender_country_nationality IN {ea} THEN 'EA' 
    ELSE 'Other' 
END AS incorporation,
SUM(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01' 
AND nominal_ccy IN ('EUR') 
AND intragroup = 1 
AND central_clearing = 'non-cleared'
{coll_eur}
GROUP BY s.business_date, lender_id, borrower_id, incorporation

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
res = check(df, ['business_date', 'incorporation'], entity='lender_id')
res.groupby('incorporation').agg(dates=('ok', 'size'), violations=('ok', lambda x: (~x).sum()), min_n_entity=('n_entity', 'min'), max_entity=('max_entity', 'max'))

In [ ]:
res[~res['ok']]

## USD

In [ ]:
query = f"""

SELECT s.business_date, lender_id, borrower_id,
CASE 
    WHEN borrower_country_nationality IN ('US') OR lender_country_nationality IN ('US') THEN 'US'  
    WHEN borrower_country_nationality IN ('GB') OR lender_country_nationality IN ('GB') THEN 'GB'  
    WHEN borrower_country_nationality IN {ea} AND lender_country_nationality IN {ea} THEN 'EA' 
    ELSE 'Other' 
END AS incorporation,
SUM(nominal_value) as value
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-07-01' 
AND nominal_ccy IN ('USD') 
AND intragroup = 1 
AND central_clearing = 'non-cleared'
{coll_usd}
GROUP BY s.business_date, lender_id, borrower_id, incorporation

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
res = check(df, ['business_date', 'incorporation'], entity='lender_id')
res.groupby('incorporation').agg(dates=('ok', 'size'), violations=('ok', lambda x: (~x).sum()), min_n_entity=('n_entity', 'min'), max_entity=('max_entity', 'max'))

In [ ]:
res[~res['ok']]